In [ ]:
import numpy as np
import pandas as pd
import joblib
import mlflow
from mlflow import sklearn
from xgboost	import XGBClassifier
from sklearn.metrics	import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
import itertools

X_train, X_test, y_train, y_test = joblib.load('../data/processed/train_test_split.pkl')

mlflow.set_experiment('credit-risk-tuning')

2026/03/17 11:21:44 INFO mlflow.tracking.fluent: Experiment with name 'credit-risk-tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/purshotam_kumar/ml_projects/credit-risk-prediction/notebooks/mlruns/2', creation_time=1773746504389, experiment_id='2', last_update_time=1773746504389, lifecycle_stage='active', name='credit-risk-tuning', tags={}, workspace='default'>

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

best_f1 = 0
best_run = None

keys = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))

print(f"Total combinations: {len(combos)}")

for combo in combos:
 params	= dict(zip(keys, combo))
  
 with mlflow.start_run(run_name=f"XGB_tuned"):
   model = XGBClassifier(**params, eval_metric='logloss', random_state=42, use_label_encoder=False)
    
   model.fit(X_train, y_train)
   y_pred = model.predict(X_test)
   y_proba = model.predict_proba(X_test)[:, 1]
    
   f1 = f1_score(y_test, y_pred)
   auc = roc_auc_score(y_test, y_proba)
   for k, v in params.items():
    mlflow.log_param(k, v)
    mlflow.log_metric('f1_score', f1)
    mlflow.log_metric('roc_auc', auc)
    mlflow.sklearn.log_model(model, "model")
    
    if f1 > best_f1:
     best_f1 = f1
     best_run = mlflow.active_run().info.run_id
     best_params = params

print(f"\nBest F1: {best_f1:.4f}")
print(f"Best params: {best_params}")
print(f"Best run ID: {best_run}")   
                                 
     
    
          

Total combinations: 108


2026/03/17 11:42:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 11:42:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/17 11:43:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 11:43:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [ ]:
best_model = XGBClassifier(**best_params, eval_metric='logloss', random_state=42)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Best tuned XGBoost:")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred))

joblib.dump(best_model, '../models/xgb_tuned.pkl')

print("Model saved to	../models/xgb_tuned.pkl")